# Analisis de prompts -- pipeline SIN malla (axial / planar)

Notebook de analisis para comparar variantes de prompt (`axis_v00`, `axis_v01`, ...,
`plane_v00`, ...) evaluadas con `Mapping/estimate_symmetry_no_mesh.py` +
`Mapping/evaluate.py` (metodos `triangulation` / `triangulation_multiplane`).

Lee los CSV combinados que genera `Mapping/compare_results_no_mesh.py --csv-dir ...`
(uno por symmetry_type, bajo `results/experiments_DD_MM_YYYY/`), los concatena entre
todas las carpetas de fecha que existan, y arma tablas de ranking + plots.

**Todas las corridas usan las mismas columnas** dentro de cada symmetry_type (las que
escribe `evaluate.py::write_csv`), asi que agregar un prompt nuevo es simplemente
correr los 3 comandos (copia -> estimate_symmetry_no_mesh -> evaluate) y volver a
correr `compare_results_no_mesh.py`; este notebook no necesita cambios.

Ver `docs/pipeline_sin_malla.md`, `docs/metricas_evaluacion.md` y
`Mapping/compare_results_no_mesh.py` para el detalle de cada metrica.

**Nota**: si corres celdas sueltas fuera de orden y algo se pisa (ej.
`PLANE_COLS` termina siendo un numero en vez de una lista), usa
`Kernel > Restart & Run All` en vez de seguir corriendo celdas sueltas.

## 0. Setup

In [ ]:
import json
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT    = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
RESULTS_ROOT = REPO_ROOT / "results"
DATA_ROOT    = Path(r"C:\Users\HP\Desktop\Seminario de Tesis I\data")
RENDERS_ROOT = DATA_ROOT / "renders"
SIZES, LIGHTINGS = [224], ["flat"]

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

plt.rcParams.update({
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 120,
})

## 1. Carga: concatena todos los `experiments_*` disponibles

Cada corrida de `compare_results_no_mesh.py --csv-dir ../results` escribe en una
carpeta `experiments_DD_MM_YYYY/` nueva (una por dia). Si volviste a evaluar el
mismo prompt+n_views otro dia, nos quedamos con la fila mas reciente
(`drop_duplicates(keep="last")` sobre `(experiment, method, n_views)`, ordenado
por fecha de carpeta).

In [ ]:
def _folder_date(p: Path):
    # experiments_DD_MM_YYYY -> tupla ordenable (YYYY, MM, DD)
    try:
        d, m, y = p.name.removeprefix("experiments_").split("_")
        return (int(y), int(m), int(d))
    except Exception:
        return (0, 0, 0)


def load_nomesh_csvs(symmetry_type: str) -> pd.DataFrame:
    """Concatena <symmetry_type>_nomesh_comparison.csv de todas las carpetas
    experiments_*/, ordenadas por fecha, y deduplica quedandose con lo mas nuevo.

    Busca tanto bajo results/ (la convencion correcta, --csv-dir ../results) como
    directamente en la raiz del repo (defensivo: si alguna corrida de
    compare_results_no_mesh.py se hizo con --csv-dir .. en vez de --csv-dir ../results
    por error, la carpeta queda mal ubicada pero igual se detecta en vez de
    ignorarse en silencio -- ya paso una vez con experiments_27_08_2026)."""
    search_roots = [RESULTS_ROOT, REPO_ROOT]
    exp_dirs = sorted(
        {d.resolve() for root in search_roots for d in root.glob("experiments_*") if d.is_dir()},
        key=_folder_date,
    )
    frames = []
    for d in exp_dirs:
        csv_path = d / f"{symmetry_type}_nomesh_comparison.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["_source_folder"] = str(d)
            frames.append(df)
    if not frames:
        raise SystemExit(
            f"[error] No se encontro ningun {symmetry_type}_nomesh_comparison.csv bajo "
            f"{RESULTS_ROOT} ni {REPO_ROOT}. Corre antes: python Mapping/compare_results_no_mesh.py "
            f"--symmetry-type {symmetry_type} --renders-root ../data/renders "
            "--csv-dir ../results --no-plots"
        )
    combined = pd.concat(frames, ignore_index=True)
    key = ["experiment", "method", "n_views"]
    combined = combined.sort_values("_source_folder").drop_duplicates(key, keep="last")
    return combined.drop(columns="_source_folder").reset_index(drop=True)


def enrich(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega columnas derivadas del nombre de experimento, utiles para agrupar/filtrar:
    - base_prompt: 'axis_v00_1_nomesh' -> 'v00'
    - is_v1: True si el prompt es la variante '_1' (mejorada)
    - is_flow: 'flowB'/'flowC' si el experimento viene de Flow B/C, sino None
    """
    df = df.copy()

    def parse(exp: str) -> tuple[str, bool, str | None]:
        stem = exp.removesuffix("_nomesh")
        flow = next((f for f in ("flowB", "flowC") if stem.endswith(f"_{f}")), None)
        if flow:
            stem = stem[: -(len(flow) + 1)]
        is_v1 = stem.endswith("_1")
        if is_v1:
            stem = stem[:-2]
        base = stem.split("_", 1)[1] if "_" in stem else stem  # 'axis_v00' -> 'v00'
        return base, is_v1, flow

    parsed = df["experiment"].apply(parse)
    df["base_prompt"] = parsed.apply(lambda t: t[0])
    df["is_v1"]       = parsed.apply(lambda t: t[1])
    df["flow"]        = parsed.apply(lambda t: t[2])
    return df

In [ ]:
df_axis  = enrich(load_nomesh_csvs("axis_sym"))
df_plane = enrich(load_nomesh_csvs("plane_sym"))

print(f"axis_sym : {df_axis['experiment'].nunique()} experimentos, {len(df_axis)} filas (experiment x n_views)")
print(f"plane_sym: {df_plane['experiment'].nunique()} experimentos, {len(df_plane)} filas")
print("\nExperimentos axis_sym :", sorted(df_axis['experiment'].unique()))
print("Experimentos plane_sym:", sorted(df_plane['experiment'].unique()))

## 2. Inventario rapido: que columnas trae cada symmetry_type

`axis_sym` (metodo `triangulation`) trae metricas angulares (angular_error, AUC,
precision@theta). `plane_sym` (metodo `triangulation_multiplane`) trae recall/precision
sobre el conjunto de planos GT -- son metricas distintas por diseno (ver
`Mapping/evaluate.py::write_csv`), no comparables entre si.

In [ ]:
def inventory(df: pd.DataFrame, label: str) -> None:
    print(f"--- {label} ---")
    print(f"columnas: {list(df.columns)}")
    print(f"n_views disponibles: {sorted(df['n_views'].unique())}")
    print(f"metodo(s): {sorted(df['method'].unique())}")
    print()

inventory(df_axis, "axis_sym")
inventory(df_plane, "plane_sym")

## 3. Ranking -- eje (`angular_error_mean` ascendente = mejor)

In [ ]:
AXIS_COLS = ["experiment", "base_prompt", "is_v1", "flow", "method", "n_views", "n_total", "n_objects",
             "angular_error_mean", "angular_error_median", "angular_error_std",
             "translation_error_mean", "translation_error_median", "translation_error_std",
             "translation_error_normalized_mean", "translation_error_normalized_median",
             "auc_angular", "n_points_mean",
             "precision_5deg", "precision_10deg", "precision_15deg",
             "sde_ref_mean", "sde_ref_min", "sde_ref_max"]
AXIS_COLS = [c for c in AXIS_COLS if c in df_axis.columns]


def best_per_experiment(df: pd.DataFrame, sort_col: str, ascending: bool,
                        cols: list[str]) -> pd.DataFrame:
    """Para cada 'experiment', se queda con la fila (n_views) que mejor puntua
    en sort_col -- 'cual es el mejor resultado que logro este prompt, en
    cualquier cantidad de vistas'."""
    assert isinstance(cols, (list, tuple)), (
        f"'cols' debe ser una lista de nombres de columna (ej. AXIS_COLS/PLANE_COLS), "
        f"recibi {cols!r} ({type(cols).__name__}). Si AXIS_COLS/PLANE_COLS quedo pisado "
        f"por accidente (ej. una celda suelta que reasigna esa variable), "
        f"hace Kernel > Restart & Run All."
    )
    idx = (df.sort_values(sort_col, ascending=ascending)
             .groupby("experiment")
             .head(1)
             .index)
    return df.loc[idx, cols].sort_values(sort_col, ascending=ascending).reset_index(drop=True)


print("Ranking por experimento (mejor n_views de cada uno), ordenado por angular_error_mean:")
_axis_ranking = best_per_experiment(df_axis, "angular_error_mean", ascending=True, cols=AXIS_COLS)
_axis_ranking

In [ ]:
_winner = _axis_ranking.iloc[0]
print(f"GANADOR eje: {_winner['experiment']}  (n_views={_winner['n_views']}, "
      f"angular_error_mean={_winner['angular_error_mean']:.2f} grados, "
      f"auc_angular={_winner['auc_angular']:.4f})")
print(f"Peor:        {_axis_ranking.iloc[-1]['experiment']}  "
      f"(angular_error_mean={_axis_ranking.iloc[-1]['angular_error_mean']:.2f} grados)")

### 3.1 Tabla completa (todos los n_views) -- eje

In [ ]:
(df_axis[AXIS_COLS]
   .sort_values(["experiment", "n_views"])
   .style.background_gradient(subset=["angular_error_mean"], cmap="RdYlGn_r")
         .background_gradient(subset=[c for c in ["auc_angular", "precision_10deg"] if c in AXIS_COLS], cmap="RdYlGn"))

### 3.2 Ablation: v0 vs v1 (eje)

Compara, para cada `base_prompt` y `n_views`, el prompt original vs. su version `_1`
mejorada -- responde "valio la pena iterar el prompt?".

In [ ]:
def plot_ablation_bar(df: pd.DataFrame, metric: str, symmetry_label: str,
                      higher_better: bool, n_views: int | None = None) -> None:
    """Barras agrupadas v0 vs v1, una por base_prompt. Si n_views es None usa
    el mayor n_views disponible para cada experimento (via best row already
    filtered by caller) -- pasar un df ya filtrado a un n_views fijo."""
    sub = df[df["flow"].isna()].copy()  # excluye flowB/flowC de esta comparacion
    if n_views is not None:
        sub = sub[sub["n_views"] == n_views]
    piv = sub.pivot_table(index="base_prompt", columns="is_v1", values=metric)
    piv = piv.rename(columns={False: "v0", True: "v1"}).sort_index()

    ax = piv.plot(kind="bar", figsize=(8, 4.5), color=["#4C72B0", "#55A868"])
    ax.set_ylabel(metric)
    direction = "mayor mejor" if higher_better else "menor mejor"
    title_nv = f"  (n_views={n_views})" if n_views is not None else ""
    ax.set_title(f"{symmetry_label} — {metric} por prompt, v0 vs v1{title_nv}  ({direction})")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


NV_AXIS = sorted(df_axis["n_views"].unique())[-1]  # el mas grande disponible
plot_ablation_bar(df_axis, "angular_error_mean", "axis_sym", higher_better=False, n_views=NV_AXIS)
if "auc_angular" in df_axis.columns:
    plot_ablation_bar(df_axis, "auc_angular", "axis_sym", higher_better=True, n_views=NV_AXIS)

### 3.3 Metricas vs n_views, una linea por experimento (eje)

In [ ]:
def plot_metric_vs_nviews(df: pd.DataFrame, metric: str, symmetry_label: str,
                          higher_better: bool) -> None:
    if metric not in df.columns:
        print(f"[skip] columna '{metric}' no existe")
        return
    exps   = sorted(df["experiment"].unique())
    cmap   = plt.get_cmap("tab20")
    colors = {e: cmap(i % 20) for i, e in enumerate(exps)}

    fig, ax = plt.subplots(figsize=(8, 5))
    for exp in exps:
        sub = df[df["experiment"] == exp].sort_values("n_views")
        if sub[metric].isna().all():
            continue
        ax.plot(sub["n_views"], sub[metric], marker="o", label=exp, color=colors[exp])
    ax.set_xlabel("n_views")
    ax.set_xticks(sorted(df["n_views"].unique()))
    ax.set_ylabel(metric)
    direction = "↑ mejor" if higher_better else "↓ mejor"
    ax.set_title(f"{symmetry_label} (sin malla) — {metric}  ({direction})")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_metric_vs_nviews(df_axis, "angular_error_mean", "axis_sym", higher_better=False)
plot_metric_vs_nviews(df_axis, "auc_angular", "axis_sym", higher_better=True)

## 4. Ranking -- plano (`recall_planes_mean` / `f1_ref` descendente = mejor)

Distinto criterio que el eje: no hay `angular_error` de un unico plano, sino
recall/precision sobre el conjunto de planos GT (metodo `triangulation_multiplane`).
Ver `Mapping/evaluate.py::evaluate_plane_multiset`.

In [ ]:
PLANE_COLS = ["experiment", "base_prompt", "is_v1", "flow", "n_views", "n_objects",
              "n_planes_predicted_mean", "n_true_planes_mean", "n_planes_matched_mean",
              "recall_planes_mean", "precision_planes_mean", "f1_ref", "f1_ref_hungarian"]
PLANE_COLS = [c for c in PLANE_COLS if c in df_plane.columns]

print("Ranking por experimento (mejor n_views de cada uno), ordenado por recall_planes_mean:")
_plane_ranking = best_per_experiment(df_plane, "recall_planes_mean", ascending=False, cols=PLANE_COLS)
_plane_ranking

In [ ]:
_winner = _plane_ranking.iloc[0]
print(f"GANADOR plano (por recall): {_winner['experiment']}  (n_views={_winner['n_views']}, "
      f"recall={_winner['recall_planes_mean']:.4f}, precision={_winner['precision_planes_mean']:.4f}, "
      f"f1_ref={_winner['f1_ref']:.4f})")
print(f"Peor:                        {_plane_ranking.iloc[-1]['experiment']}  "
      f"(recall={_plane_ranking.iloc[-1]['recall_planes_mean']:.4f}, "
      f"f1_ref={_plane_ranking.iloc[-1]['f1_ref']:.4f})")

# Ranking alternativo ordenado por f1_ref (combina recall+precision) -- no siempre
# coincide con el orden por recall_planes_mean solo.
print("\nRanking alternativo por f1_ref (mejor n_views de cada uno):")
best_per_experiment(df_plane, "f1_ref", ascending=False, cols=PLANE_COLS)

### 4.1 Tabla completa (todos los n_views) -- plano

In [ ]:
grad_cols = [c for c in ["recall_planes_mean", "precision_planes_mean", "f1_ref"] if c in PLANE_COLS]
(df_plane[PLANE_COLS]
   .sort_values(["experiment", "n_views"])
   .style.background_gradient(subset=grad_cols, cmap="RdYlGn"))

### 4.2 Ablation: v0 vs v1 (plano)

In [ ]:
NV_PLANE = 14  # n_views=14 fue el optimo empirico observado, no el maximo
if NV_PLANE not in df_plane["n_views"].unique():
    NV_PLANE = sorted(df_plane["n_views"].unique())[-1]

plot_ablation_bar(df_plane, "recall_planes_mean", "plane_sym", higher_better=True, n_views=NV_PLANE)
if "f1_ref" in df_plane.columns:
    plot_ablation_bar(df_plane, "f1_ref", "plane_sym", higher_better=True, n_views=NV_PLANE)

### 4.3 Metricas vs n_views, una linea por experimento (plano)

Ojo: a diferencia del eje, mas vistas NO siempre mejora recall/precision en el
flujo sin malla (el metodo multi-plano puede sobre-segmentar con muchas vistas) --
esta es la seccion para confirmarlo/refutarlo con cada tanda nueva de prompts.

In [ ]:
plot_metric_vs_nviews(df_plane, "recall_planes_mean", "plane_sym", higher_better=True)
plot_metric_vs_nviews(df_plane, "precision_planes_mean", "plane_sym", higher_better=True)
plot_metric_vs_nviews(df_plane, "n_planes_predicted_mean", "plane_sym", higher_better=None)

## 5. Filtrar a un subconjunto de prompts nuevos

Cuando agregues una tanda nueva de prompts y quieras comparar SOLO esos (sin el
ruido visual de todos los anteriores), filtra por `experiment` antes de graficar.

In [ ]:
NUEVOS_AXIS  = ["axis_v06_nomesh", "axis_v07_nomesh"]
NUEVOS_PLANE = ["plane_v06_nomesh", "plane_v07_nomesh"]

if NUEVOS_AXIS:
    sub = df_axis[df_axis["experiment"].isin(NUEVOS_AXIS)]
    display(sub[AXIS_COLS].sort_values(["experiment", "n_views"]))
    plot_metric_vs_nviews(sub, "angular_error_mean", "axis_sym (nuevos)", higher_better=False)

if NUEVOS_PLANE:
    sub = df_plane[df_plane["experiment"].isin(NUEVOS_PLANE)]
    display(sub[PLANE_COLS].sort_values(["experiment", "n_views"]))
    plot_metric_vs_nviews(sub, "recall_planes_mean", "plane_sym (nuevos)", higher_better=True)

## 6. Exportar ranking final

In [ ]:
out_dir = RESULTS_ROOT / f"experiments_{date.today().strftime('%d_%m_%Y')}"
out_dir.mkdir(parents=True, exist_ok=True)

best_per_experiment(df_axis, "angular_error_mean", ascending=True, cols=AXIS_COLS) \
    .to_csv(out_dir / "axis_sym_nomesh_ranking.csv", index=False)
best_per_experiment(df_plane, "recall_planes_mean", ascending=False, cols=PLANE_COLS) \
    .to_csv(out_dir / "plane_sym_nomesh_ranking.csv", index=False)

print(f"Guardado en: {out_dir}")

## 7. Estadisticas completas por objeto (min / mean / median / max / std)

Los CSV combinados (seccion 1) solo traen `..._mean`/`..._median`/`..._std` --
**no traen min/max** de `angular_error`/`translation_error`/`recall_planes`/
`precision_planes` (`Mapping/evaluate.py::compute_summary` no los calcula para
esas columnas, solo para `sde_ref`). Para tenerlos hay que leer directo los
`eval_..._results.json` por objeto (los mismos que usa `plot_precision_curve`
en `Mapping/compare_results_no_mesh.py`) y calcular las 5 estadisticas ahi.

Esto es clave para el punto que vimos en la conversacion: una `..._mean` alta
con una `..._median` baja y un `..._std` grande es la firma de un puñado de
outliers, no de un problema sistematico -- `min`/`max` lo confirman de un
vistazo (si `max` es muchisimo mas grande que `median`, hay outliers).

In [ ]:
def _results_json_path(symmetry_type: str, experiment_id: str, method: str) -> Path:
    size_tag  = "s" + "_".join(str(s) for s in SIZES)
    light_tag = "_".join(LIGHTINGS)
    return RENDERS_ROOT / symmetry_type / f"eval_{size_tag}_{light_tag}_{experiment_id}_{method}_results.json"


def load_object_level_metric(symmetry_type: str, experiment_id: str, method: str,
                             value_key: str) -> pd.DataFrame:
    """Lee eval_..._results.json (por objeto, por n_views) y devuelve un DataFrame
    largo: experiment, n_views, object_id, value -- solo para status=='ok'.
    value_key: 'angular_error_deg'/'translation_error' (axis/plane simple) o
    'recall_planes'/'precision_planes' (plane multiplane)."""
    path = _results_json_path(symmetry_type, experiment_id, method)
    if not path.exists():
        return pd.DataFrame(columns=["experiment", "n_views", "object_id", "value"])
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for obj_id, per_nv in data.get("objects", {}).items():
        if per_nv is None:
            continue
        for nv_key, m in per_nv.items():
            if isinstance(m, dict) and m.get("status") == "ok" and m.get(value_key) is not None:
                rows.append({"experiment": experiment_id, "n_views": int(nv_key),
                            "object_id": obj_id, "value": m[value_key]})
    return pd.DataFrame(rows)


def full_stats(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """min/mean/median/max/std agrupado por experiment x n_views."""
    if df.empty:
        print(f"[skip] sin datos para {label}")
        return df
    stats = (df.groupby(["experiment", "n_views"])["value"]
                .agg(["min", "mean", "median", "max", "std", "count"])
                .round(4)
                .reset_index()
                .sort_values(["experiment", "n_views"]))
    stats.insert(2, "metric", label)
    return stats

### 7.1 Eje -- `angular_error_deg` y `translation_error` (min/mean/median/max/std)

In [ ]:
axis_angular_obj = pd.concat(
    [load_object_level_metric("axis_sym", exp, "triangulation", "angular_error_deg")
     for exp in df_axis["experiment"].unique()],
    ignore_index=True,
)
axis_trans_obj = pd.concat(
    [load_object_level_metric("axis_sym", exp, "triangulation", "translation_error")
     for exp in df_axis["experiment"].unique()],
    ignore_index=True,
)

axis_full_stats = pd.concat([
    full_stats(axis_angular_obj, "angular_error_deg"),
    full_stats(axis_trans_obj,   "translation_error"),
], ignore_index=True)

# 'max' >> 'median' con 'std' grande = outliers dominando la media (ver conversacion)
axis_full_stats["outlier_ratio"] = (axis_full_stats["max"] / axis_full_stats["median"]).round(1)
axis_full_stats.sort_values(["metric", "experiment", "n_views"])

### 7.2 Plano -- `recall_planes` y `precision_planes` (min/mean/median/max/std)

Analogo, pero sobre las metricas de `triangulation_multiplane` (no hay
`angular_error`/`translation_error` de un unico plano en este metodo -- ver
la diferencia `triangulation` vs `triangulation_multiplane` que vimos antes).

In [ ]:
plane_recall_obj = pd.concat(
    [load_object_level_metric("plane_sym", exp, "triangulation_multiplane", "recall_planes")
     for exp in df_plane["experiment"].unique()],
    ignore_index=True,
)
plane_precision_obj = pd.concat(
    [load_object_level_metric("plane_sym", exp, "triangulation_multiplane", "precision_planes")
     for exp in df_plane["experiment"].unique()],
    ignore_index=True,
)

plane_full_stats = pd.concat([
    full_stats(plane_recall_obj,    "recall_planes"),
    full_stats(plane_precision_obj, "precision_planes"),
], ignore_index=True)

plane_full_stats.sort_values(["metric", "experiment", "n_views"])

In [ ]:
axis_full_stats.to_csv(out_dir / "axis_sym_nomesh_full_stats.csv", index=False)
plane_full_stats.to_csv(out_dir / "plane_sym_nomesh_full_stats.csv", index=False)
print(f"Guardado: {out_dir / 'axis_sym_nomesh_full_stats.csv'}")
print(f"Guardado: {out_dir / 'plane_sym_nomesh_full_stats.csv'}")